# PLN na Prática: Domando Dados Textuais com Pandas

**Hands-on adaptado** para trabalhar com um dataset textual sintético de **prontuários ambulatoriais**, com classificação de gravidade em:
- **verde**
- **amarelo**
- **vermelho**

## Objetivo
Neste notebook, o participante vai aprender a:
- carregar e inspecionar um dataset textual realista;
- explorar textos clínicos com Pandas;
- limpar e padronizar dados textuais;
- criar atributos úteis para PLN;
- preparar os dados para tarefas futuras, como classificação, extração de informação e uso com LLMs.

## Contexto
O dataset contém registros sintéticos de pacientes atendidos no ambulatório.  
Cada linha possui:
- um identificador do registro;
- um texto de prontuário;
- uma categoria de gravidade clínica.


## 1. Setup e Ingestão de Dados Textuais

Vamos começar carregando o dataset sintético de prontuários.


In [ ]:
import pandas as pd
import numpy as np
import re

pd.set_option("display.max_colwidth", 220)


In [ ]:
# Carregando o dataset
df = pd.read_csv("dataset_prontuarios_sinteticos_1000.csv", encoding="utf-8")

df.head()
# df.sample()


,id,categoria,texto
661,662,verde,"Paciente apresenta quadro de rinite/alergia com coriza e espirros, sem febre, sem dispneia e com exame físico estável. Prescrito tratamento sintomático e orientação de retorno."
666,667,amarelo,"Paciente diabético refere glicemias elevadas nas últimas 24 horas, com poliúria e mal-estar, porém sem rebaixamento do nível de consciência. Quadro inspira atenção apesar da estabilidade inicial. Paciente possui ante..."
593,594,amarelo,"Paciente com episódio de vômitos e diarreia desde ontem, sinais de desidratação leve a moderada e necessidade de observação clínica. Permanece estável, porém demanda reavaliação em curto prazo. Sinais vitais registra..."


## 2. Anatomia de um DataFrame Textual

Nesta etapa, queremos entender:
- o tamanho do dataset;
- os tipos de dados;
- a presença de valores nulos;
- a distribuição das categorias.


In [ ]:
df.shape

(1000, 3)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   id         1000 non-null   int64 
 1   categoria  1000 non-null   object
 2   texto      1000 non-null   object
dtypes: int64(1), object(2)
memory usage: 23.6+ KB


In [ ]:
df.dtypes

,0
id,int64
categoria,object
texto,object


In [ ]:
df.isna().sum()

,0
id,0
categoria,0
texto,0


## 3. Exploração Inicial de Dados Textuais

Vamos observar as primeiras linhas e entender como os prontuários estão estruturados.


In [ ]:
df.head(10)

,id,categoria,texto
0,1,amarelo,"Paciente com dor abdominal moderada iniciada há 24 horas, náuseas ocasionais e redução do apetite. Sem instabilidade hemodinâmica até o momento. Permanece estável, porém demanda reavaliação em curto prazo. Paciente p..."
1,2,verde,"Paciente em retorno de ferida superficial em cicatrização adequada, sem secreção purulenta, sem febre e sem dor intensa. Mantém-se comunicativo, orientado e sem sinais de alarme. Sinais vitais registrados na triagem ..."
2,3,verde,"Paciente refere dor lombar leve após esforço físico, sem déficit neurológico, sem alteração esfincteriana e deambulando sem auxílio. Prescrito tratamento sintomático e orientação de retorno. Sinais vitais registrados..."
3,4,verde,"Paciente relata dor de garganta leve há 3 dias, sem febre no momento, aceitando dieta e hidratando bem. Exame ambulatorial sem sinais de gravidade. Prescrito tratamento sintomático e orientação de retorno. Sinais vit..."
4,5,amarelo,"Paciente refere pico hipertensivo em domicílio com cefaleia e tontura, no momento vigil, orientado e sem déficit focal evidente. Requer monitorização e investigação complementar. Sinais vitais registrados na triagem ..."
5,6,amarelo,"Paciente com dor abdominal moderada iniciada há 12 horas, náuseas ocasionais e redução do apetite. Sem instabilidade hemodinâmica até o momento. Quadro inspira atenção apesar da estabilidade inicial."
6,7,amarelo,"Paciente com crise de ansiedade importante, taquicardia e sensação de falta de ar, mas mantendo saturação preservada e boa resposta ao acolhimento. Mantido em observação ambulatorial prioritária. Refere início dos si..."
7,8,verde,"Paciente com queixa de gastralgia leve pós-prandial, sem vômitos persistentes, sem sinais de sangramento digestivo e em bom estado geral. Conduta expectante e acompanhamento ambulatorial. Sinais vitais registrados na..."
8,9,verde,"Paciente apresenta quadro de rinite/alergia com coriza e espirros, sem febre, sem dispneia e com exame físico estável. Sem necessidade de encaminhamento imediato."
9,10,verde,"Paciente refere prurido cutâneo localizado, sem lesões extensas, sem sinais infecciosos e sem comprometimento sistêmico. Mantém-se comunicativo, orientado e sem sinais de alarme."


In [ ]:
df.sample(10, random_state=42)

,id,categoria,texto
521,522,verde,"Paciente com queixa de gastralgia leve pós-prandial, sem vômitos persistentes, sem sinais de sangramento digestivo e em bom estado geral. Conduta expectante e acompanhamento ambulatorial. Refere início dos sintomas a..."
737,738,verde,"Paciente com dor articular leve crônica, sem edema importante, sem limitação funcional acentuada e sem sinais inflamatórios exuberantes. Prescrito tratamento sintomático e orientação de retorno."
740,741,amarelo,"Paciente diabético refere glicemias elevadas nas últimas 24 horas, com poliúria e mal-estar, porém sem rebaixamento do nível de consciência. Requer monitorização e investigação complementar."
660,661,verde,"Paciente refere dor lombar leve após esforço físico, sem déficit neurológico, sem alteração esfincteriana e deambulando sem auxílio. Sem necessidade de encaminhamento imediato. Sinais vitais registrados na triagem e ..."
411,412,amarelo,"Paciente apresenta celulite em membro inferior com hiperemia, dor local e aumento de temperatura, sem sinais de choque séptico. Quadro inspira atenção apesar da estabilidade inicial. Sinais vitais registrados na tria..."
678,679,verde,"Paciente refere dor lombar leve após esforço físico, sem déficit neurológico, sem alteração esfincteriana e deambulando sem auxílio. Prescrito tratamento sintomático e orientação de retorno."
626,627,verde,"Paciente relata dor de garganta leve há 4 dias, sem febre no momento, aceitando dieta e hidratando bem. Exame ambulatorial sem sinais de gravidade. Mantém-se comunicativo, orientado e sem sinais de alarme."
513,514,verde,"Paciente com tosse seca há 7 dias, afebril, sem desconforto respiratório e saturando bem em ar ambiente. Conduta ambulatorial e orientações domiciliares. Conduta expectante e acompanhamento ambulatorial."
859,860,amarelo,"Paciente apresenta celulite em membro inferior com hiperemia, dor local e aumento de temperatura, sem sinais de choque séptico. Quadro inspira atenção apesar da estabilidade inicial. Paciente possui antecedente de asma."
136,137,verde,"Paciente refere prurido cutâneo localizado, sem lesões extensas, sem sinais infecciosos e sem comprometimento sistêmico. Sem necessidade de encaminhamento imediato. Sinais vitais registrados na triagem e reavaliados ..."


In [ ]:
df["categoria"].value_counts()

,count
categoria,
verde,580
amarelo,312
vermelho,108


In [ ]:
df["categoria"].value_counts(normalize=True).round(3)

,proportion
categoria,
verde,0.580
amarelo,0.312
vermelho,0.108


### Alguns exemplos de prontuários por categoria


In [ ]:
for cat in ["verde", "amarelo", "vermelho"]:
    print("="*80)
    print(f"CATEGORIA: {cat.upper()}")
    exemplos = df[ df["categoria"] == cat ]["texto"].head(3).tolist()
    for i, texto in enumerate(exemplos, start=1):
        print(f"\nExemplo {i}: {texto}")


CATEGORIA: VERDE

Exemplo 1: Paciente em retorno de ferida superficial em cicatrização adequada, sem secreção purulenta, sem febre e sem dor intensa. Mantém-se comunicativo, orientado e sem sinais de alarme. Sinais vitais registrados na triagem e reavaliados após 20 minutos. Paciente possui antecedente de diabetes mellitus.

Exemplo 2: Paciente refere dor lombar leve após esforço físico, sem déficit neurológico, sem alteração esfincteriana e deambulando sem auxílio. Prescrito tratamento sintomático e orientação de retorno. Sinais vitais registrados na triagem e reavaliados após 15 minutos. Refere início dos sintomas após uso de medicação recente.

Exemplo 3: Paciente relata dor de garganta leve há 3 dias, sem febre no momento, aceitando dieta e hidratando bem. Exame ambulatorial sem sinais de gravidade. Prescrito tratamento sintomático e orientação de retorno. Sinais vitais registrados na triagem e reavaliados após 15 minutos. Paciente possui antecedente de hipertensão. Refere início d

## 4. Seleção e Filtragem com foco em texto

Aqui vamos recuperar subconjuntos clinicamente relevantes.


In [ ]:
# Apenas casos vermelhos
df[df["categoria"] == "vermelho"].head()


,id,categoria,texto
24,25,vermelho,"Paciente com rebaixamento do nível de consciência e resposta lenta a comandos, acompanhado por familiar. Necessita intervenção imediata. Caso comunicado imediatamente à equipe médica."
33,34,vermelho,"Paciente apresenta dispneia importante em repouso, uso de musculatura acessória e saturação reduzida em ar ambiente. Caso grave, prioridade máxima. Caso comunicado imediatamente à equipe médica."
53,54,vermelho,"Paciente com hiperglicemia grave sintomática, vômitos persistentes, desidratação importante e rebaixamento do estado geral. Transporte interno e manejo urgente realizados. Sinais vitais registrados na triagem e reava..."
60,61,vermelho,"Paciente admitido no ambulatório com dor torácica intensa súbita há 20 minutos, sudorese fria, palidez e queda do estado geral. Encaminhamento imediato para atendimento de emergência. Paciente não deve aguardar em fi..."
71,72,vermelho,"Paciente apresenta dispneia importante em repouso, uso de musculatura acessória e saturação reduzida em ar ambiente. Caso grave, prioridade máxima. Caso comunicado imediatamente à equipe médica."


In [ ]:
# Prontuários que mencionam dor torácica
df[ df["texto"].str.contains("dor torácica")].head(10)


,id,categoria,texto
37,38,amarelo,"Paciente relata dor torácica atípica de intensidade moderada, sem sudorese fria importante e sem irradiação típica, porém requer investigação no ambulatório. Permanece estável, porém demanda reavaliação em curto praz..."
60,61,vermelho,"Paciente admitido no ambulatório com dor torácica intensa súbita há 20 minutos, sudorese fria, palidez e queda do estado geral. Encaminhamento imediato para atendimento de emergência. Paciente não deve aguardar em fi..."
74,75,amarelo,"Paciente relata dor torácica atípica de intensidade moderada, sem sudorese fria importante e sem irradiação típica, porém requer investigação no ambulatório. Mantido em observação ambulatorial prioritária."
91,92,amarelo,"Paciente relata dor torácica atípica de intensidade moderada, sem sudorese fria importante e sem irradiação típica, porém requer investigação no ambulatório. Necessita exames e conduta breve pela equipe."
127,128,vermelho,"Paciente admitido no ambulatório com dor torácica intensa súbita há 10 minutos, sudorese fria, palidez e queda do estado geral. Encaminhamento imediato para atendimento de emergência. Transporte interno e manejo urge..."
133,134,vermelho,"Paciente admitido no ambulatório com dor torácica intensa súbita há 10 minutos, sudorese fria, palidez e queda do estado geral. Encaminhamento imediato para atendimento de emergência. Equipe acionada para suporte ime..."
147,148,amarelo,"Paciente relata dor torácica atípica de intensidade moderada, sem sudorese fria importante e sem irradiação típica, porém requer investigação no ambulatório. Requer monitorização e investigação complementar."
180,181,amarelo,"Paciente relata dor torácica atípica de intensidade moderada, sem sudorese fria importante e sem irradiação típica, porém requer investigação no ambulatório. Permanece estável, porém demanda reavaliação em curto praz..."
198,199,vermelho,"Paciente admitido no ambulatório com dor torácica intensa súbita há 20 minutos, sudorese fria, palidez e queda do estado geral. Encaminhamento imediato para atendimento de emergência. Equipe acionada para suporte ime..."
243,244,vermelho,"Paciente admitido no ambulatório com dor torácica intensa súbita há 20 minutos, sudorese fria, palidez e queda do estado geral. Encaminhamento imediato para atendimento de emergência. Transporte interno e manejo urge..."


In [ ]:
# Termos clínicos de alerta
padrao_alerta = r"dispneia|rebaixamento|sangramento|convulsiva|avc|cianose|hipotensão|urgente|emergência"

df[df["texto"].str.contains(padrao_alerta, case=False, na=False, regex=True)].head(15)


,id,categoria,texto
7,8,verde,"Paciente com queixa de gastralgia leve pós-prandial, sem vômitos persistentes, sem sinais de sangramento digestivo e em bom estado geral. Conduta expectante e acompanhamento ambulatorial. Sinais vitais registrados na..."
8,9,verde,"Paciente apresenta quadro de rinite/alergia com coriza e espirros, sem febre, sem dispneia e com exame físico estável. Sem necessidade de encaminhamento imediato."
16,17,verde,"Paciente com queixa de gastralgia leve pós-prandial, sem vômitos persistentes, sem sinais de sangramento digestivo e em bom estado geral. Prescrito tratamento sintomático e orientação de retorno."
17,18,verde,"Paciente apresenta quadro de rinite/alergia com coriza e espirros, sem febre, sem dispneia e com exame físico estável. Sem necessidade de encaminhamento imediato."
18,19,amarelo,"Paciente comparece com crise asmática leve a moderada, apresentando sibilância e dispneia aos esforços, com melhora parcial após medicação habitual. Permanece estável, porém demanda reavaliação em curto prazo. Sinais..."
19,20,verde,"Paciente com queixa de gastralgia leve pós-prandial, sem vômitos persistentes, sem sinais de sangramento digestivo e em bom estado geral. Sem necessidade de encaminhamento imediato. Sinais vitais registrados na triag..."
21,22,amarelo,"Paciente diabético refere glicemias elevadas nas últimas 24 horas, com poliúria e mal-estar, porém sem rebaixamento do nível de consciência. Mantido em observação ambulatorial prioritária. Sinais vitais registrados n..."
24,25,vermelho,"Paciente com rebaixamento do nível de consciência e resposta lenta a comandos, acompanhado por familiar. Necessita intervenção imediata. Caso comunicado imediatamente à equipe médica."
25,26,verde,"Paciente com queixa de gastralgia leve pós-prandial, sem vômitos persistentes, sem sinais de sangramento digestivo e em bom estado geral. Mantém-se comunicativo, orientado e sem sinais de alarme. Paciente possui ante..."
32,33,verde,"Paciente apresenta quadro de rinite/alergia com coriza e espirros, sem febre, sem dispneia e com exame físico estável. Prescrito tratamento sintomático e orientação de retorno. Sinais vitais registrados na triagem e ..."


## 5. Limpeza de Dados Textuais

Em PLN, normalmente criamos uma versão tratada do texto para análise computacional.

Vamos:
- preservar o texto original;
- converter para minúsculas;
- remover pontuação;
- normalizar espaços.


In [ ]:
df["texto_original"] = df["texto"]

df["texto_limpo"] = (
    df["texto"]
    .fillna("")
    .str.lower()
    .str.replace(r"[^\w\sÀ-ÿ]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df[["texto_original", "texto_limpo"]].head(10)


,texto_original,texto_limpo
0,"Paciente com dor abdominal moderada iniciada há 24 horas, náuseas ocasionais e redução do apetite. Sem instabilidade hemodinâmica até o momento. Permanece estável, porém demanda reavaliação em curto prazo. Paciente p...",paciente com dor abdominal moderada iniciada há 24 horas náuseas ocasionais e redução do apetite sem instabilidade hemodinâmica até o momento permanece estável porém demanda reavaliação em curto prazo paciente possui...
1,"Paciente em retorno de ferida superficial em cicatrização adequada, sem secreção purulenta, sem febre e sem dor intensa. Mantém-se comunicativo, orientado e sem sinais de alarme. Sinais vitais registrados na triagem ...",paciente em retorno de ferida superficial em cicatrização adequada sem secreção purulenta sem febre e sem dor intensa mantém se comunicativo orientado e sem sinais de alarme sinais vitais registrados na triagem e rea...
2,"Paciente refere dor lombar leve após esforço físico, sem déficit neurológico, sem alteração esfincteriana e deambulando sem auxílio. Prescrito tratamento sintomático e orientação de retorno. Sinais vitais registrados...",paciente refere dor lombar leve após esforço físico sem déficit neurológico sem alteração esfincteriana e deambulando sem auxílio prescrito tratamento sintomático e orientação de retorno sinais vitais registrados na ...
3,"Paciente relata dor de garganta leve há 3 dias, sem febre no momento, aceitando dieta e hidratando bem. Exame ambulatorial sem sinais de gravidade. Prescrito tratamento sintomático e orientação de retorno. Sinais vit...",paciente relata dor de garganta leve há 3 dias sem febre no momento aceitando dieta e hidratando bem exame ambulatorial sem sinais de gravidade prescrito tratamento sintomático e orientação de retorno sinais vitais r...
4,"Paciente refere pico hipertensivo em domicílio com cefaleia e tontura, no momento vigil, orientado e sem déficit focal evidente. Requer monitorização e investigação complementar. Sinais vitais registrados na triagem ...",paciente refere pico hipertensivo em domicílio com cefaleia e tontura no momento vigil orientado e sem déficit focal evidente requer monitorização e investigação complementar sinais vitais registrados na triagem e re...
5,"Paciente com dor abdominal moderada iniciada há 12 horas, náuseas ocasionais e redução do apetite. Sem instabilidade hemodinâmica até o momento. Quadro inspira atenção apesar da estabilidade inicial.",paciente com dor abdominal moderada iniciada há 12 horas náuseas ocasionais e redução do apetite sem instabilidade hemodinâmica até o momento quadro inspira atenção apesar da estabilidade inicial
6,"Paciente com crise de ansiedade importante, taquicardia e sensação de falta de ar, mas mantendo saturação preservada e boa resposta ao acolhimento. Mantido em observação ambulatorial prioritária. Refere início dos si...",paciente com crise de ansiedade importante taquicardia e sensação de falta de ar mas mantendo saturação preservada e boa resposta ao acolhimento mantido em observação ambulatorial prioritária refere início dos sintom...
7,"Paciente com queixa de gastralgia leve pós-prandial, sem vômitos persistentes, sem sinais de sangramento digestivo e em bom estado geral. Conduta expectante e acompanhamento ambulatorial. Sinais vitais registrados na...",paciente com queixa de gastralgia leve pós prandial sem vômitos persistentes sem sinais de sangramento digestivo e em bom estado geral conduta expectante e acompanhamento ambulatorial sinais vitais registrados na tri...
8,"Paciente apresenta quadro de rinite/alergia com coriza e espirros, sem febre, sem dispneia e com exame físico estável. Sem necessidade de encaminhamento imediato.",paciente apresenta quadro de rinite alergia com coriza e espirros sem febre sem dispneia e com exame físico estável sem necessidade de encaminhamento imediato
9,"Paciente refere prurido cutâneo localizado, sem lesões extensas, sem sinais infecciosos e sem comprometimento sistêmico. Mantém-se comu

### Comparando antes e depois da limpeza


In [ ]:
df.loc[:9, ["categoria", "texto_original", "texto_limpo"]]


,categoria,texto_original,texto_limpo
0,amarelo,"Paciente com dor abdominal moderada iniciada há 24 horas, náuseas ocasionais e redução do apetite. Sem instabilidade hemodinâmica até o momento. Permanece estável, porém demanda reavaliação em curto prazo. Paciente p...",paciente com dor abdominal moderada iniciada há 24 horas náuseas ocasionais e redução do apetite sem instabilidade hemodinâmica até o momento permanece estável porém demanda reavaliação em curto prazo paciente possui...
1,verde,"Paciente em retorno de ferida superficial em cicatrização adequada, sem secreção purulenta, sem febre e sem dor intensa. Mantém-se comunicativo, orientado e sem sinais de alarme. Sinais vitais registrados na triagem ...",paciente em retorno de ferida superficial em cicatrização adequada sem secreção purulenta sem febre e sem dor intensa mantém se comunicativo orientado e sem sinais de alarme sinais vitais registrados na triagem e rea...
2,verde,"Paciente refere dor lombar leve após esforço físico, sem déficit neurológico, sem alteração esfincteriana e deambulando sem auxílio. Prescrito tratamento sintomático e orientação de retorno. Sinais vitais registrados...",paciente refere dor lombar leve após esforço físico sem déficit neurológico sem alteração esfincteriana e deambulando sem auxílio prescrito tratamento sintomático e orientação de retorno sinais vitais registrados na ...
3,verde,"Paciente relata dor de garganta leve há 3 dias, sem febre no momento, aceitando dieta e hidratando bem. Exame ambulatorial sem sinais de gravidade. Prescrito tratamento sintomático e orientação de retorno. Sinais vit...",paciente relata dor de garganta leve há 3 dias sem febre no momento aceitando dieta e hidratando bem exame ambulatorial sem sinais de gravidade prescrito tratamento sintomático e orientação de retorno sinais vitais r...
4,amarelo,"Paciente refere pico hipertensivo em domicílio com cefaleia e tontura, no momento vigil, orientado e sem déficit focal evidente. Requer monitorização e investigação complementar. Sinais vitais registrados na triagem ...",paciente refere pico hipertensivo em domicílio com cefaleia e tontura no momento vigil orientado e sem déficit focal evidente requer monitorização e investigação complementar sinais vitais registrados na triagem e re...
5,amarelo,"Paciente com dor abdominal moderada iniciada há 12 horas, náuseas ocasionais e redução do apetite. Sem instabilidade hemodinâmica até o momento. Quadro inspira atenção apesar da estabilidade inicial.",paciente com dor abdominal moderada iniciada há 12 horas náuseas ocasionais e redução do apetite sem instabilidade hemodinâmica até o momento quadro inspira atenção apesar da estabilidade inicial
6,amarelo,"Paciente com crise de ansiedade importante, taquicardia e sensação de falta de ar, mas mantendo saturação preservada e boa resposta ao acolhimento. Mantido em observação ambulatorial prioritária. Refere início dos si...",paciente com crise de ansiedade importante taquicardia e sensação de falta de ar mas mantendo saturação preservada e boa resposta ao acolhimento mantido em observação ambulatorial prioritária refere início dos sintom...
7,verde,"Paciente com queixa de gastralgia leve pós-prandial, sem vômitos persistentes, sem sinais de sangramento digestivo e em bom estado geral. Conduta expectante e acompanhamento ambulatorial. Sinais vitais registrados na...",paciente com queixa de gastralgia leve pós prandial sem vômitos persistentes sem sinais de sangramento digestivo e em bom estado geral conduta expectante e acompanhamento ambulatorial sinais vitais registrados na tri...
8,verde,"Paciente apresenta quadro de rinite/alergia com coriza e espirros, sem febre, sem dispneia e com exame físico estável. Sem necessidade de encaminhamento imediato.",paciente apresenta quadro de rinite alergia com coriza e espirros sem febre sem dispneia e com exame físico estável sem necessidade de encaminhamento imediato
9,verde,"Paciente refere prurido cutâneo localizado, sem lesões ex

## 6. Engenharia de Atributos Textuais com Pandas

Agora vamos gerar atributos úteis para análise de textos clínicos:
- número de caracteres;
- número de palavras;
- número de dígitos;
- presença de termos de alerta;
- presença de comorbidades mencionadas;
- indicador de texto vazio.


In [ ]:
df["num_caracteres"] = df["texto_limpo"].str.len()
df["num_palavras"] = df["texto_limpo"].str.split().apply(len)
df["num_digitos"] = df["texto_limpo"].str.count(r"\d")
df["tem_alerta"] = df["texto_limpo"].str.contains(padrao_alerta, case=False, na=False, regex=True)

padrao_comorbidades = r"hipertensão|diabetes mellitus|asma|ansiedade|enxaqueca|gastrite"
df["tem_comorbidade_mencionada"] = df["texto_limpo"].str.contains(padrao_comorbidades, case=False, na=False, regex=True)

df.head()


,id,categoria,texto,texto_original,texto_limpo,num_caracteres,num_palavras,num_digitos,tem_alerta,tem_comorbidade_mencionada
0,1,amarelo,"Paciente com dor abdominal moderada iniciada há 24 horas, náuseas ocasionais e redução do apetite. Sem instabilidade hemodinâmica até o momento. Permanece estável, porém demanda reavaliação em curto prazo. Paciente p...","Paciente com dor abdominal moderada iniciada há 24 horas, náuseas ocasionais e redução do apetite. Sem instabilidade hemodinâmica até o momento. Permanece estável, porém demanda reavaliação em curto prazo. Paciente p...",paciente com dor abdominal moderada iniciada há 24 horas náuseas ocasionais e redução do apetite sem instabilidade hemodinâmica até o momento permanece estável porém demanda reavaliação em curto prazo paciente possui...,243,34,2,False,True
1,2,verde,"Paciente em retorno de ferida superficial em cicatrização adequada, sem secreção purulenta, sem febre e sem dor intensa. Mantém-se comunicativo, orientado e sem sinais de alarme. Sinais vitais registrados na triagem ...","Paciente em retorno de ferida superficial em cicatrização adequada, sem secreção purulenta, sem febre e sem dor intensa. Mantém-se comunicativo, orientado e sem sinais de alarme. Sinais vitais registrados na triagem ...",paciente em retorno de ferida superficial em cicatrização adequada sem secreção purulenta sem febre e sem dor intensa mantém se comunicativo orientado e sem sinais de alarme sinais vitais registrados na triagem e rea...,289,43,2,False,True
2,3,verde,"Paciente refere dor lombar leve após esforço físico, sem déficit neurológico, sem alteração esfincteriana e deambulando sem auxílio. Prescrito tratamento sintomático e orientação de retorno. Sinais vitais registrados...","Paciente refere dor lombar leve após esforço físico, sem déficit neurológico, sem alteração esfincteriana e deambulando sem auxílio. Prescrito tratamento sintomático e orientação de retorno. Sinais vitais registrados...",paciente refere dor lombar leve após esforço físico sem déficit neurológico sem alteração esfincteriana e deambulando sem auxílio prescrito tratamento sintomático e orientação de retorno sinais vitais registrados na ...,310,44,2,False,False
3,4,verde,"Paciente relata dor de garganta leve há 3 dias, sem febre no momento, aceitando dieta e hidratando bem. Exame ambulatorial sem sinais de gravidade. Prescrito tratamento sintomático e orientação de retorno. Sinais vit...","Paciente relata dor de garganta leve há 3 dias, sem febre no momento, aceitando dieta e hidratando bem. Exame ambulatorial sem sinais de gravidade. Prescrito tratamento sintomático e orientação de retorno. Sinais vit...",paciente relata dor de garganta leve há 3 dias sem febre no momento aceitando dieta e hidratando bem exame ambulatorial sem sinais de gravidade prescrito tratamento sintomático e orientação de retorno sinais vitais r...,358,53,3,False,True
4,5,amarelo,"Paciente refere pico hipertensivo em domicílio com cefaleia e tontura, no momento vigil, orientado e sem déficit focal evidente. Requer monitorização e investigação complementar. Sinais vitais registrados na triagem ...","Paciente refere pico hipertensivo em domicílio com cefaleia e tontura, no momento vigil, orientado e sem déficit focal evidente. Requer monitorização e investigação complementar. Sinais vitais registrados na triagem ...",paciente refere pico hipertensivo em domicílio com cefaleia e tontura no momento vigil orientado e sem déficit focal evidente requer monitorização e investigação complementar sinais vitais registrados na triagem e re...,241,34,2,False,False


In [ ]:
df[["num_caracteres", "num_palavras", "num_digitos"]].describe().round(2)


,num_caracteres,num_palavras,num_digitos
count,1000.000,1000.000,1000.000
mean,225.273,32.064,1.002
std,44.108,6.680,1.116
min,144.000,17.000,0.000
25%,189.000,27.000,0.000
50%,223.500,32.000,1.000
75%,256.000,37.000,2.000
max,370.000,57.000,4.000


In [ ]:
df.groupby("categoria")[["num_caracteres", "num_palavras"]].mean().round(2)


,num_caracteres,num_palavras
categoria,,
amarelo,230.48,32.77
verde,223.92,32.19
vermelho,217.52,29.32


### Tokenização simples com Pandas


In [ ]:
df["tokens"] = df["texto_limpo"].str.split()

df[["texto_limpo", "tokens"]].head(5)


,texto_limpo,tokens
0,paciente com dor abdominal moderada iniciada há 24 horas náuseas ocasionais e redução do apetite sem instabilidade hemodinâmica até o momento permanece estável porém demanda reavaliação em curto prazo paciente possui...,"[paciente, com, dor, abdominal, moderada, iniciada, há, 24, horas, náuseas, ocasionais, e, redução, do, apetite, sem, instabilidade, hemodinâmica, até, o, momento, permanece, estável, porém, demanda, reavaliação, em,..."
1,paciente em retorno de ferida superficial em cicatrização adequada sem secreção purulenta sem febre e sem dor intensa mantém se comunicativo orientado e sem sinais de alarme sinais vitais registrados na triagem e rea...,"[paciente, em, retorno, de, ferida, superficial, em, cicatrização, adequada, sem, secreção, purulenta, sem, febre, e, sem, dor, intensa, mantém, se, comunicativo, orientado, e, sem, sinais, de, alarme, sinais, vitais..."
2,paciente refere dor lombar leve após esforço físico sem déficit neurológico sem alteração esfincteriana e deambulando sem auxílio prescrito tratamento sintomático e orientação de retorno sinais vitais registrados na ...,"[paciente, refere, dor, lombar, leve, após, esforço, físico, sem, déficit, neurológico, sem, alteração, esfincteriana, e, deambulando, sem, auxílio, prescrito, tratamento, sintomático, e, orientação, de, retorno, sin..."
3,paciente relata dor de garganta leve há 3 dias sem febre no momento aceitando dieta e hidratando bem exame ambulatorial sem sinais de gravidade prescrito tratamento sintomático e orientação de retorno sinais vitais r...,"[paciente, relata, dor, de, garganta, leve, há, 3, dias, sem, febre, no, momento, aceitando, dieta, e, hidratando, bem, exame, ambulatorial, sem, sinais, de, gravidade, prescrito, tratamento, sintomático, e, orientaç..."
4,paciente refere pico hipertensivo em domicílio com cefaleia e tontura no momento vigil orientado e sem déficit focal evidente requer monitorização e investigação complementar sinais vitais registrados na triagem e re...,"[paciente, refere, pico, hipertensivo, em, domicílio, com, cefaleia, e, tontura, no, momento, vigil, orientado, e, sem, déficit, focal, evidente, requer, monitorização, e, investigação, complementar, sinais, vitais, ..."


In [ ]:
tokens_frequentes = (
    df["tokens"]
    .explode()
    .dropna()
    .value_counts()
    .head(30)
)

tokens_frequentes


,count
tokens,
e,2264
de,1727
sem,1686
paciente,1275
sinais,957
com,711
após,600
em,580
vitais,412


## 7. Conversão e Padronização de Tipos

Vamos garantir consistência para o pipeline.


In [ ]:
df["texto_limpo"] = df["texto_limpo"].astype("string")
df["categoria"] = pd.Categorical(df["categoria"], categories=["verde", "amarelo", "vermelho"], ordered=True)
df["tem_alerta"] = df["tem_alerta"].astype(bool)
df["tem_comorbidade_mencionada"] = df["tem_comorbidade_mencionada"].astype(bool)

df.dtypes


,0
id,int64
categoria,category
texto,object
texto_original,object
texto_limpo,string[python]
num_caracteres,int64
num_palavras,int64
num_digitos,int64
tem_alerta,bool
tem_comorbidade_mencionada,bool


## 8. Análises rápidas orientadas a PLN

Vamos responder perguntas úteis:
1. Qual categoria tende a ter textos maiores?
2. Casos vermelhos possuem mais termos de alerta?
3. Quais palavras aparecem com frequência nos prontuários?


In [ ]:
df.groupby("categoria")["num_palavras"].mean().sort_values(ascending=False)


/tmp/ipykernel_499/819649692.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("categoria")["num_palavras"].mean().sort_values(ascending=False)


,num_palavras
categoria,
amarelo,32.769231
verde,32.194828
vermelho,29.324074


In [ ]:
df.groupby("categoria")["tem_alerta"].mean().round(3)


/tmp/ipykernel_499/2515026927.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("categoria")["tem_alerta"].mean().round(3)


,tem_alerta
categoria,
verde,0.176
amarelo,0.234
vermelho,0.917


### Casos potencialmente mais críticos segundo regras simples


In [ ]:
casos_criticos = df[
    (df["categoria"] == "vermelho") |
    ((df["categoria"] == "amarelo") & (df["tem_alerta"]))
]

casos_criticos[["id", "categoria", "texto_original", "tem_alerta"]].head(20)


,id,categoria,texto_original,tem_alerta
18,19,amarelo,"Paciente comparece com crise asmática leve a moderada, apresentando sibilância e dispneia aos esforços, com melhora parcial após medicação habitual. Permanece estável, porém demanda reavaliação em curto prazo. Sinais...",True
21,22,amarelo,"Paciente diabético refere glicemias elevadas nas últimas 24 horas, com poliúria e mal-estar, porém sem rebaixamento do nível de consciência. Mantido em observação ambulatorial prioritária. Sinais vitais registrados n...",True
24,25,vermelho,"Paciente com rebaixamento do nível de consciência e resposta lenta a comandos, acompanhado por familiar. Necessita intervenção imediata. Caso comunicado imediatamente à equipe médica.",True
33,34,vermelho,"Paciente apresenta dispneia importante em repouso, uso de musculatura acessória e saturação reduzida em ar ambiente. Caso grave, prioridade máxima. Caso comunicado imediatamente à equipe médica.",True
36,37,amarelo,"Paciente diabético refere glicemias elevadas nas últimas 24 horas, com poliúria e mal-estar, porém sem rebaixamento do nível de consciência. Requer monitorização e investigação complementar.",True
40,41,amarelo,"Paciente comparece com crise asmática leve a moderada, apresentando sibilância e dispneia aos esforços, com melhora parcial após medicação habitual. Quadro inspira atenção apesar da estabilidade inicial.",True
53,54,vermelho,"Paciente com hiperglicemia grave sintomática, vômitos persistentes, desidratação importante e rebaixamento do estado geral. Transporte interno e manejo urgente realizados. Sinais vitais registrados na triagem e reava...",True
54,55,amarelo,"Paciente diabético refere glicemias elevadas nas últimas 24 horas, com poliúria e mal-estar, porém sem rebaixamento do nível de consciência. Quadro inspira atenção apesar da estabilidade inicial.",True
55,56,amarelo,"Paciente comparece com crise asmática leve a moderada, apresentando sibilância e dispneia aos esforços, com melhora parcial após medicação habitual. Quadro inspira atenção apesar da estabilidade inicial.",True
60,61,vermelho,"Paciente admitido no ambulatório com dor torácica intensa súbita há 20 minutos, sudorese fria, palidez e queda do estado geral. Encaminhamento imediato para atendimento de emergência. Paciente não deve aguardar em fi...",True


In [ ]:
casos_criticos.shape

(181, 11)

In [ ]:
casos_criticos["categoria"].value_counts()

,count
categoria,
vermelho,108
amarelo,73
verde,0


In [ ]:
df["categoria"].value_counts()

,count
categoria,
verde,580
amarelo,312
vermelho,108


## 9. Preparando dados para etapas futuras com IA

Uma etapa comum é gerar uma coluna pronta para entrada em modelos.


In [ ]:
df["entrada_modelo"] = (
    "Classifique a gravidade clínica deste prontuário ambulatorial. "
    "Texto do prontuário: " + df["texto_limpo"].astype(str)
)

df[["id", "categoria", "entrada_modelo"]].head(5)


,id,categoria,entrada_modelo
0,1,amarelo,Classifique a gravidade clínica deste prontuário ambulatorial. Texto do prontuário: paciente com dor abdominal moderada iniciada há 24 horas náuseas ocasionais e redução do apetite sem instabilidade hemodinâmica até ...
1,2,verde,Classifique a gravidade clínica deste prontuário ambulatorial. Texto do prontuário: paciente em retorno de ferida superficial em cicatrização adequada sem secreção purulenta sem febre e sem dor intensa mantém se comu...
2,3,verde,Classifique a gravidade clínica deste prontuário ambulatorial. Texto do prontuário: paciente refere dor lombar leve após esforço físico sem déficit neurológico sem alteração esfincteriana e deambulando sem auxílio pr...
3,4,verde,Classifique a gravidade clínica deste prontuário ambulatorial. Texto do prontuário: paciente relata dor de garganta leve há 3 dias sem febre no momento aceitando dieta e hidratando bem exame ambulatorial sem sinais d...
4,5,amarelo,Classifique a gravidade clínica deste prontuário ambulatorial. Texto do prontuário: paciente refere pico hipertensivo em domicílio com cefaleia e tontura no momento vigil orientado e sem déficit focal evidente requer...


## 10. Exportando o dataset tratado


In [ ]:
colunas_saida = [
    "id",
    "categoria",
    "texto_original",
    "texto_limpo",
    "num_caracteres",
    "num_palavras",
    "num_digitos",
    "tem_alerta",
    "tem_comorbidade_mencionada",
    "entrada_modelo",
]

df[colunas_saida].to_csv("dataset_prontuarios_tratado.csv", index=False, encoding="utf-8")
df[colunas_saida].head()


,id,categoria,texto_original,texto_limpo,num_caracteres,num_palavras,num_digitos,tem_alerta,tem_comorbidade_mencionada,entrada_modelo
0,1,amarelo,"Paciente com dor abdominal moderada iniciada há 24 horas, náuseas ocasionais e redução do apetite. Sem instabilidade hemodinâmica até o momento. Permanece estável, porém demanda reavaliação em curto prazo. Paciente p...",paciente com dor abdominal moderada iniciada há 24 horas náuseas ocasionais e redução do apetite sem instabilidade hemodinâmica até o momento permanece estável porém demanda reavaliação em curto prazo paciente possui...,243,34,2,False,True,Classifique a gravidade clínica deste prontuário ambulatorial. Texto do prontuário: paciente com dor abdominal moderada iniciada há 24 horas náuseas ocasionais e redução do apetite sem instabilidade hemodinâmica até ...
1,2,verde,"Paciente em retorno de ferida superficial em cicatrização adequada, sem secreção purulenta, sem febre e sem dor intensa. Mantém-se comunicativo, orientado e sem sinais de alarme. Sinais vitais registrados na triagem ...",paciente em retorno de ferida superficial em cicatrização adequada sem secreção purulenta sem febre e sem dor intensa mantém se comunicativo orientado e sem sinais de alarme sinais vitais registrados na triagem e rea...,289,43,2,False,True,Classifique a gravidade clínica deste prontuário ambulatorial. Texto do prontuário: paciente em retorno de ferida superficial em cicatrização adequada sem secreção purulenta sem febre e sem dor intensa mantém se comu...
2,3,verde,"Paciente refere dor lombar leve após esforço físico, sem déficit neurológico, sem alteração esfincteriana e deambulando sem auxílio. Prescrito tratamento sintomático e orientação de retorno. Sinais vitais registrados...",paciente refere dor lombar leve após esforço físico sem déficit neurológico sem alteração esfincteriana e deambulando sem auxílio prescrito tratamento sintomático e orientação de retorno sinais vitais registrados na ...,310,44,2,False,False,Classifique a gravidade clínica deste prontuário ambulatorial. Texto do prontuário: paciente refere dor lombar leve após esforço físico sem déficit neurológico sem alteração esfincteriana e deambulando sem auxílio pr...
3,4,verde,"Paciente relata dor de garganta leve há 3 dias, sem febre no momento, aceitando dieta e hidratando bem. Exame ambulatorial sem sinais de gravidade. Prescrito tratamento sintomático e orientação de retorno. Sinais vit...",paciente relata dor de garganta leve há 3 dias sem febre no momento aceitando dieta e hidratando bem exame ambulatorial sem sinais de gravidade prescrito tratamento sintomático e orientação de retorno sinais vitais r...,358,53,3,False,True,Classifique a gravidade clínica deste prontuário ambulatorial. Texto do prontuário: paciente relata dor de garganta leve há 3 dias sem febre no momento aceitando dieta e hidratando bem exame ambulatorial sem sinais d...
4,5,amarelo,"Paciente refere pico hipertensivo em domicílio com cefaleia e tontura, no momento vigil, orientado e sem déficit focal evidente. Requer monitorização e investigação complementar. Sinais vitais registrados na triagem ...",paciente refere pico hipertensivo em domicílio com cefaleia e tontura no momento vigil orientado e sem déficit focal evidente requer monitorização e investigação complementar sinais vitais registrados na triagem e re...,241,34,2,False,False,Classifique a gravidade clínica deste prontuário ambulatorial. Texto do prontuário: paciente refere pico hipertensivo em domicílio com cefaleia e tontura no momento vigil orientado e sem déficit focal evidente requer...


## 11. Desafios para o aluno

### Desafio 1
Crie uma coluna chamada `tem_dor` que marque os prontuários que mencionam a palavra **dor**.

### Desafio 2
Filtre apenas os casos **amarelos** que mencionam **febre**.

### Desafio 3
Monte uma tabela com:
- quantidade de registros por categoria;
- média de palavras por categoria;
- percentual de registros com termos de alerta.

### Desafio 4
Crie uma coluna `prioridade_regra` com a lógica:
- **alta**: vermelho;
- **media**: amarelo;
- **baixa**: verde.

### Desafio 5
Conte quantos prontuários mencionam cada uma das comorbidades abaixo:
- hipertensão
- diabetes mellitus
- asma
- ansiedade
- enxaqueca
- gastrite


## 12. Próximos passos na trilha

Este notebook já deixa o dataset pronto para:
- classificação automática de risco;
- análise exploratória de textos clínicos;
- extração de informação;
- embeddings;
- experimentos com LLMs.

---
**Mensagem final:** mesmo em aplicações de IA na saúde, a qualidade do pipeline com Pandas continua sendo a base de tudo.
